# 🎯 KPI métier & segmentation RFM — NordRetail

**Objectif** : calculer les indicateurs clés (CA, panier moyen, marge, évolution mensuelle) puis
segmenter les clients avec la méthode **RFM** (Récence, Fréquence, Montant).

> **Analogie** — un KPI, c'est le **tableau de bord de la voiture** : quelques cadrans qui disent
> l'essentiel d'un coup d'œil. La RFM, c'est **trier ses clients comme on trie son courrier** :
> les fidèles récents d'un côté, les endormis de l'autre.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = "../../99-Brief/Data-Analyst/data"

In [ ]:
faits = pd.read_csv(f"{DATA}/Faits_Ventes.csv")
dates = pd.read_csv(f"{DATA}/Dim_Date.csv")
df = faits.merge(dates[["date_id","date","annee","mois","nom_mois"]], on="date_id", how="left")
df["date"] = pd.to_datetime(df["date"])
df.head()

## 1. Les KPI globaux

In [ ]:
ca_total     = df["montant"].sum()
panier_moyen = df["montant"].mean()
marge_total  = df["marge"].sum()
taux_marge   = marge_total / ca_total
nb_ventes    = len(df)
nb_clients   = df["client_id"].nunique()
print(f"CA total       : {ca_total:,.0f} €")
print(f"Panier moyen   : {panier_moyen:,.2f} €")
print(f"Marge totale   : {marge_total:,.0f} € ({taux_marge:.1%})")
print(f"Nb de ventes   : {nb_ventes:,}")
print(f"Nb de clients  : {nb_clients:,}")

## 2. Évolution mensuelle du CA

In [ ]:
mensuel = df.groupby(["annee","mois"])["montant"].sum().reset_index()
mensuel["periode"] = mensuel["annee"].astype(str) + "-" + mensuel["mois"].astype(str).str.zfill(2)
mensuel.plot(x="periode", y="montant", figsize=(11,4), marker="o", title="CA mensuel (€)", legend=False)
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 3. Segmentation RFM
Pour chaque client : **Récence** (jours depuis son dernier achat), **Fréquence** (nb d'achats),
**Montant** (total dépensé). On note chaque axe de 1 à 4 par quartiles.

In [ ]:
date_ref = df["date"].max() + pd.Timedelta(days=1)  # lendemain du dernier achat observé
rfm = df.groupby("client_id").agg(
    recence=("date", lambda d: (date_ref - d.max()).days),
    frequence=("vente_id", "count"),
    montant=("montant", "sum"),
).reset_index()
rfm.head()

In [ ]:
# Scores 1-4 (récence : plus c'est petit, mieux c'est -> on inverse)
rfm["R"] = pd.qcut(rfm["recence"], 4, labels=[4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequence"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["M"] = pd.qcut(rfm["montant"], 4, labels=[1,2,3,4]).astype(int)
rfm["score_RFM"] = rfm["R"] + rfm["F"] + rfm["M"]
rfm.head()

## 4. Nommer les segments

In [ ]:
def segment(row):
    if row["score_RFM"] >= 10: return "Champions"
    if row["R"] >= 3 and row["F"] >= 3: return "Fidèles"
    if row["R"] <= 2 and row["M"] >= 3: return "À réactiver (gros)"
    if row["R"] <= 2: return "Endormis"
    return "Occasionnels"
rfm["segment"] = rfm.apply(segment, axis=1)
rfm["segment"].value_counts()

In [ ]:
rfm["segment"].value_counts().plot(kind="barh", figsize=(8,4), title="Répartition des segments clients")
plt.tight_layout(); plt.show()

## 🎯 À toi de jouer
Quel est le **montant moyen dépensé** par segment ? Les « Champions » dépensent-ils vraiment plus ?

In [ ]:
# Écris ta réponse ici :


<details><summary>💡 Corrigé</summary>

```python
rfm.groupby('segment')['montant'].mean().sort_values(ascending=False)
```
</details>

In [ ]:
rfm.groupby("segment")["montant"].mean().sort_values(ascending=False).round(0)

## ✅ À retenir
- Un bon KPI est **simple, daté et comparable** (CA, panier moyen, taux de marge).
- La **RFM** transforme un historique d'achats en **segments actionnables** pour le marketing.
- `qcut` découpe en quartiles ; la récence se score **à l'envers** (récent = bon).